# Hyperliquid Perpetual Futures: Liquidation & P&L Calculator


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Hyperliquid Perpetual Futures: Liquidation & P&L Calculator

This notebook helps beginners understand how liquidation prices, margin requirements, and profit/loss work on perpetual futures exchanges like Hyperliquid.

You'll learn:
- How to calculate your liquidation price
- How P&L changes as the market moves
- How leverage amplifies both gains and losses

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def calculate_liquidation_price(entry_price, leverage, position_type='long', maintenance_margin=0.05):
    """
    Calculate liquidation price for a perpetual futures position.
    
    Args:
        entry_price: Price at which you entered the position
        leverage: Leverage used (e.g., 10 for 10x)
        position_type: 'long' or 'short'
        maintenance_margin: Maintenance margin ratio (typically 0.05 for 5%)
    
    Returns:
        Liquidation price
    """
    # Initial margin = 1 / leverage
    initial_margin = 1 / leverage
    
    if position_type.lower() == 'long':
        # For long: liquidation when price drops such that losses exceed maintenance margin
        # liquidation_price = entry_price * (1 - maintenance_margin)
        liquidation_price = entry_price * (1 - maintenance_margin)
    else:  # short
        # For short: liquidation when price rises
        # liquidation_price = entry_price * (1 + maintenance_margin)
        liquidation_price = entry_price * (1 + maintenance_margin)
    
    return liquidation_price

def calculate_pnl(entry_price, current_price, position_size, leverage, position_type='long'):
    """
    Calculate unrealized P&L for a perpetual position.
    
    Args:
        entry_price: Entry price
        current_price: Current market price
        position_size: Size of position (in base asset units)
        leverage: Leverage used
        position_type: 'long' or 'short'
    
    Returns:
        P&L in USDT and percentage
    """
    if position_type.lower() == 'long':
        price_change = current_price - entry_price
    else:  # short
        price_change = entry_price - current_price
    
    # P&L = position_size * price_change * leverage
    pnl_usd = position_size * price_change * leverage
    
    # Initial margin cost
    margin_used = (position_size * entry_price) / leverage
    
    # P&L percentage
    pnl_percent = (pnl_usd / margin_used) * 100
    
    return pnl_usd, pnl_percent

print("Functions defined. Ready to calculate liquidation prices and P&L.")

## Example 1: Long Bitcoin at 10x Leverage

Let's say you:
- Enter a long position on BTC at $45,000
- Use 10x leverage
- Position size: 1 BTC
- Maintenance margin: 5%

What is your liquidation price?

In [ ]:
# Example parameters
entry_price = 45000
leverage = 10
position_type = 'long'
position_size = 1  # 1 BTC
maintenance_margin = 0.05

# Calculate liquidation price
liq_price = calculate_liquidation_price(entry_price, leverage, position_type, maintenance_margin)

print(f"Entry Price: ${entry_price:,.2f}")
print(f"Leverage: {leverage}x")
print(f"Liquidation Price: ${liq_price:,.2f}")
print(f"Distance to Liquidation: {((entry_price - liq_price) / entry_price * 100):.2f}%")
print(f"\nIf BTC drops {((entry_price - liq_price) / entry_price * 100):.2f}%, you get liquidated.")

## P&L Analysis: How Your Position Changes with Market Price

Now let's see how your profit or loss changes as BTC moves up or down from your entry price.

In [ ]:
# Create a range of prices from -20% to +20% of entry price
price_range = np.linspace(entry_price * 0.80, entry_price * 1.20, 100)
pnl_values = []

for price in price_range:
    pnl_usd, pnl_pct = calculate_pnl(entry_price, price, position_size, leverage, position_type)
    pnl_values.append(pnl_usd)

# Plot P&L curve
plt.figure(figsize=(12, 6))
plt.plot(price_range, pnl_values, linewidth=2.5, label='P&L', color='blue')

# Mark entry price
plt.axvline(entry_price, color='green', linestyle='--', label='Entry Price', linewidth=2)

# Mark liquidation price
plt.axvline(liq_price, color='red', linestyle='--', label='Liquidation Price', linewidth=2)

# Mark break-even if applicable
plt.axhline(0, color='gray', linestyle='-', alpha=0.3)

plt.xlabel('BTC Price (USDT)', fontsize=12)
plt.ylabel('Unrealized P&L (USDT)', fontsize=12)
plt.title(f'P&L Curve for 1 BTC Long at {leverage}x Leverage (Entry: ${entry_price:,.0f})', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Chart shows P&L at different BTC prices.")
print(f"Red line = liquidation price (${liq_price:,.2f})")
print(f"Green line = your entry price (${entry_price:,.2f})")

In [ ]:
# Detailed P&L at specific price points
test_prices = [40000, 42500, 45000, 47500, 50000]
print("\n" + "="*70)
print(f"{'Price':<12} {'P&L (USD)':<15} {'P&L (%)':<12} {'Status':<15}")
print("="*70)

for price in test_prices:
    pnl_usd, pnl_pct = calculate_pnl(entry_price, price, position_size, leverage, position_type)
    if price <= liq_price:
        status = 'LIQUIDATED'
    elif pnl_usd > 0:
        status = 'Profit'
    elif pnl_usd < 0:
        status = 'Loss'
    else:
        status = 'Break-even'
    
    print(f"${price:<11,.0f} ${pnl_usd:<14,.2f} {pnl_pct:<11.2f}% {status:<15}")

print("="*70)

## Key Takeaways

1. **Liquidation Price**: With 10x leverage, a 5% price drop liquidates you
2. **Leverage Amplification**: Your P&L is multiplied by your leverage (good for wins, bad for losses)
3. **Risk Management**: Always know your liquidation price before entering a position
4. **Margin**: The amount of capital you actually put down (here: $45,000 / 10 = $4,500)

---

## Reference

[protraderdaily](https://protraderdaily.com/crypto/hyperliquid-perpetual-futures-trading-for-beginners)
